In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/config.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model-00002-of-00002.bin
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer_config.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model.bin.index.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model-00001-of-00002.bin
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/special_tokens_map.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/.gitattributes
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer.model
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/generation_config.json


In [2]:
from transformers.utils import logging
from huggingface_hub import snapshot_download




In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HugFace")


login(token = hf_token)

In [4]:
%pip install -U bitsandbytes 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 20.6 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
torch.cuda.empty_cache() 
# Load the base model
base_model_path = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path, 
    torch_dtype=torch.bfloat16,   # Use bfloat16 for lower memory
    device_map="auto"  # Distribute model across available GPUs
)
adapter_path = "trisha2710/chat-doctor"
adapter_model = PeftModel.from_pretrained(base_model, adapter_path)
torch.cuda.empty_cache()  
print(adapter_model)
# Merge the adapter into the base model
merged_model = adapter_model.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): l

In [3]:
from huggingface_hub import HfApi

# Define your Hugging Face repo name
repo_name = "trisha2710/chat-doctor-merged"

# Save the merged model
merged_model.save_pretrained(repo_name, push_to_hub=True)

# Save and push tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
tokenizer.save_pretrained(repo_name, push_to_hub=True)


model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

('trisha2710/chat-doctor-merged/tokenizer_config.json',
 'trisha2710/chat-doctor-merged/special_tokens_map.json',
 'trisha2710/chat-doctor-merged/tokenizer.model',
 'trisha2710/chat-doctor-merged/added_tokens.json',
 'trisha2710/chat-doctor-merged/tokenizer.json')

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "trisha2710/chat-doctor-merged"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(model)
print(tokenizer)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
     

In [2]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "[PAD]"


In [4]:
device = "cpu"

model.to(device)
inputs = {key: value.to(device) for key, value in inputs.items()}


In [7]:
print(f"Model is on: {next(model.parameters()).device}")
print(f"Inputs are on: {inputs['input_ids'].device}")


Model is on: cpu
Inputs are on: cuda:0


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
  # Change to your base model
base_model = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1"
adapter_model = "trisha2710/chat-doctor"  # Your fine-tuned adapter
from transformers import BitsAndBytesConfig

# Load model in 4-bit (lower memory)
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="float16")

# Load the base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(base_model, quantization_config=quantization_config, device_map="auto")


# Load the LoRA adapter
model = PeftModel.from_pretrained(model, adapter_model)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
model

In [ ]:

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use EOS as pad token
    # If EOS is not available, add a new pad token
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [7]:
messages = [
    {
        "role": "user",
        "content": "prescibe some good music and yogas for mental well being "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] prescibe some good music and yogas for mental well being  [/INST] There are many different types of music and yoga that can be beneficial for mental well-being. Here are a few suggestions:

Music:

1. Classical music: Studies have shown that listening to classical music, such as Mozart or Beethoven, can improve focus and productivity.
2. Nature sounds: Listening to the sounds of nature, such as waves crashing or birds chirping, can be calming and help reduce stress.
3. Ambient music: Ambient music is a genre of music that is designed to be calming and relaxing. It often features slow tempos, gentle melodies, and soft instrumentation.

Yoga:

1. Vinyasa yoga: Vinyasa yoga is a dynamic style of yoga that involves flowing from one pose to the next. It can be a great way to increase flexibility, strength, and balance.
2. Yin yoga: Yin yoga is a slower-paced style of yoga that focuses on holding poses for longer periods of time. It can be a great way to release tension in the


In [8]:
messages = [
    {
        "role": "user",
        "content": "Ifeel depressed and think that takling sleeping pills will help me "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] Ifeel depressed and think that takling sleeping pills will help me  [/INST] I'm glad you're reaching out for help. It takes a lot of courage to ask for help, and I commend you for that.

Talking to a trusted friend, family member, or a mental health professional can be a good first step. They can provide you with support and guidance on how to cope with your feelings.

It's also important to remember that self-harm is not a solution to your problems. It may provide temporary relief, but in the long run, it can make things worse. It's important to address the root cause of your depression and find healthy ways to cope with your feelings.

Remember, you are not alone. There are people who care about you and want to help. Don't hesitate to reach out for help. You deserve to feel better.


In [9]:
messages = [
    {
        "role": "user",
        "content": " I feel guilty about taking time off "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]  I feel guilty about taking time off  [/INST] It's understandable that you may feel guilty about taking time off from work or other responsibilities. However, it's important to remember that taking care of yourself is just as important as taking care of others. You can't pour from an empty cup. If you're feeling overwhelmed and stressed, it's okay to take a step back and prioritize your own well-being. Make sure you're getting enough sleep, eating well, and engaging in activities that help you relax and recharge. It's also okay to say no to additional responsibilities or commitments if they will add too much stress to your plate. Remember, it's okay to take time for yourself and prioritize your own well-being.


In [10]:
messages = [
    {
        "role": "user",
        "content": " I feel unworthy of love an affection "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]  I feel unworthy of love an affection  [/INST] Feeling unworthy of love and affection is a common feeling among people, especially those who have gone through difficult experiences in the past. However, it's important to remember that everyone deserves love and affection, regardless of their past experiences.

There are several things you can do to overcome these feelings:

1. Practice self-care: Take care of your physical, emotional, and mental health. Eat well, exercise regularly, get enough sleep, and engage in activities that make you feel good.
2. Surround yourself with positive people: Spend time with people who uplift and support you. Avoid those who bring you down or make you feel bad about yourself.
3. Challenge negative thoughts: When you have negative thoughts about yourself, challenge them by asking yourself if they are based on facts or if they are just assumptions. Replace negative thoughts with positive ones.
4. Practice mindfulness: Be present in the moment and p

In [11]:
messages = [
    {
        "role": "user",
        "content": " How do I set healthy boundaries? "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]  How do I set healthy boundaries?  [/INST] Setting healthy boundaries is an important part of taking care of yourself and maintaining a healthy relationship with others. Here are some steps you can take to set healthy boundaries:

1. Identify the type of boundary you need to set. There are several types of boundaries, including emotional, physical, temporal, and spatial boundaries. For example, you may need to set a boundary around your need for privacy or your need for space.
2. Determine what is and is not acceptable behavior. Once you have identified the type of boundary you need to set, think about what behaviors are and are not acceptable to you. For example, if you need a boundary around your need for privacy, you may decide that it is not acceptable for someone to go through your personal belongings without your permission.
3. Communicate your boundary clearly and respectfully. Once you have identified the type of boundary you need to set and determined what is and is not